# 第8章 数えずに学ぶ：ニューラルbigram

確率の表を勾配降下法で「学習」させたら、数えた答えに近づいていった

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：bigramの確率表を、数えるのではなく勾配降下法で学習するニューラルネットワーク（最小のニューラル言語モデル）
- **身につく数学・理論**：ロジットの表＝パラメータ ／ 交差エントロピー損失（`F.cross_entropy`） ／ ミニバッチ確率的勾配降下法（SGD） ／ 最尤推定：損失最小化は「数える」と同じ答えになる ／ 学習曲線の読み方 ／ 正則化とスムージングの関係
- **新しく使うPython・PyTorch**：`torch.randint` でランダムに選ぶ ／ `torch.Generator` で乱数を固定 ／ `F.cross_entropy` ／ `W.grad = None`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 60〜90分 |
| 難易度 | ★★★☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第6章の「確率の表」と、第7章の「坂を下る方法」を合体させます。表の中身をでたらめ（全部0）から始めて、損失が小さくなる方向に少しずつ調整していくと……表の中身が、**数えた答えにだんだん近づいていく**はずです。これが、世界で一番小さな **ニューラル言語モデル** です。

> **🔗 これまでの復習とつながり：第3章・第5章・第6章・第7章とのつながり**
>
> これまで別々に学んできた4つの道具が、この章でついに1本の線につながります！
> - **第3章（softmax）**：生の点数（ロジット）を、合計1の確率分布に変える変換。
> - **第5章（交差エントロピー損失）**：正解の文字に付けた確率の $-\log$ の平均。まっさらな一様分布なら損失は $\log V$。
> - **第6章（one-hot × 行列）**：one-hot ベクトルに表 `W` を掛ける計算は、行を取り出す `W[x]` と数学的にまったく同じ。
> - **第7章（勾配降下法と自動微分）**：`loss.backward()` で勾配を求め、`with torch.no_grad():` の中で `W -= lr * W.grad` と坂を下り、勾配をリセットする。

**ミナト**：数えれば一瞬で答えが出るのに、わざわざ学習させる意味あるんですか？

**ボンゴ先生**：鋭い。この章のモデルだけ見れば、意味はない。でも、「数える」は直前の文字が1つのときしか使えない。直前10文字を見たいときは数えられない（第4章の組み合わせ爆発）。でも「学習」なら、どんなに複雑なモデルにも使える。今日は、**答えが分かっている問題**で、学習という方法が本当に正しい答えにたどり着くかを確かめるんだ。答え合わせ付きの練習問題だね。

## 準備

準備セルに、今回は文字⇔番号の変換と、データのテンソル化も入れておきます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
import torch
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
print(V, train_data.shape, val_data.shape)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
2061 torch.Size([146220]) torch.Size([16247])
```

</details>

今度は文章全体を**数字のテンソル**にしてしまいました。ここからは、文字ではなく番号で考えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
xs, ys = train_data[:-1], train_data[1:]     # 入力（今の文字）と正解（次の文字）
vxs, vys = val_data[:-1], val_data[1:]
print("入力：", xs[:8].tolist())
print("正解：", ys[:8].tolist())
print("文字で見ると：", [itos[i] for i in xs[:8].tolist()], "→", [itos[i] for i in ys[:8].tolist()])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
入力： [138, 3, 263, 1272, 45, 1375, 0, 0]
正解： [3, 263, 1272, 45, 1375, 0, 0, 132]
文字で見ると： ['上', '\u3000', '先', '生', 'と', '私', '\n', '\n'] → ['\u3000', '先', '生', 'と', '私', '\n', '\n', '一']
```

</details>

入力と正解が、1文字ずつずれて並んでいます。「`上` の次は `　`（全角空白）」「`先` の次は `生`」……。これが、言語モデルの **教師データ** です。問題（入力）と答え（正解）の組が14万個以上あります。

> **💡 理論のポイント：ラベルシフトと「教師強制（teacher forcing）」**
>
> `xs = train_data[:-1]`、`ys = train_data[1:]` のように、同じ文章を1文字だけ右にずらして入力と正解の組を作る手法を **ラベルシフト** と呼びます。そして学習時に、モデル自身の前の予測結果ではなく**毎回「本当の正解の文字」を入力として与えて次を予測させる**方法を **教師強制**（teacher forcing）と呼びます。
>
> - **なぜ教師強制を使うのか？**：学習初期のモデルはでたらめな文字ばかり出すため、自分の出力を次の入力にすると連鎖的に崩壊して学習が進みません。正解を毎回教えながら練習させることで、全位置の損失を安定かつ並列に計算できます。
> - **生成（推論）との違い**：第3章の自己回帰生成では正解が存在しないため、モデル自身の予測を次の入力へ戻します。この「学習時は正解入力、本番は自己出力」というズレは研究でも重要なテーマです（第11章の複数位置同時学習でもそのまま使います）。
>
> ※ なお、ここでは第5章と同様に簡潔さのため全テキスト `text` から語彙 `chars` を作っていますが、実務・研究では検証データからの情報漏れを防ぐため訓練データのみで語彙を構築します。

#### 📝 確認問題（`ch08-q01` · Lv.2 · 1文字ずらし教師データ xs = data[:-1], ys = data[1:] の仕組み）

文章のID列から「今の文字 `xs`」と「次の文字 `ys`」のペアを作るスライスを確認します。次のコードを実行すると何が表示されますか？

```python
data = [10, 20, 30, 40]
xs = data[:-1]
ys = data[1:]
print(xs, ys)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Python のスライス `[:-1]` と `[1:]` がそれぞれリストのどちらの端を1要素ずつ削るか思い出しましょう。
- **ヒント 2**：`data[:-1]` は末尾の `40` を除いた先頭3要素、`data[1:]` は先頭の `10` を除いた後ろ3要素になります。
- **期待される出力**：`[10, 20, 30] [20, 30, 40]`

**考え方**：1文字ずらして並べるだけで、長さ $N$ の文章から $N - 1$ 組の「入力 → 正解」ペアを一瞬で作れます。

1. `data[:-1]` → 末尾の `40` を除いた `[10, 20, 30]`。
1. `data[1:]` → 先頭の `10` を除いた `[20, 30, 40]`。
1. ペアは `(10→20), (20→30), (30→40)` の3組になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q01

</details>

## モデル：中身がでたらめな表

第6章の確率の表 `P` の代わりに、**点数（ロジット）の表** `W` を用意します。中身は最初、全部0です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
W = torch.zeros((V, V), requires_grad=True)
print(W.shape, "パラメータの数：", W.numel())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2061, 2061]) パラメータの数： 4247721
```

</details>

使い方は第6章と同じです。今の文字の番号で行を取り出せば、それが「次の文字の点数」。第3章のsoftmaxで確率にします。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
x = torch.tensor([stoi["先"]])
logits = W[x]                       # 「先」の行＝次の文字の点数（ロジット）
probs = F.softmax(logits, dim=1)
print(logits.shape, probs.shape)
print("「生」の確率：", probs[0, stoi["生"]].item(), " ＝ 1/V：", 1 / V)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([1, 2061]) torch.Size([1, 2061])
「生」の確率： 0.0004852013662457466  ＝ 1/V： 0.00048520135856380397
```

</details>

点数が全部0なので、softmaxするとどの文字も同じ確率 $1/2061$ になります。何も知らない、まっさらな状態です。

<details><summary><b>基礎の補足：なぜ `W[x]` は「最小のニューラルネットワーク」と言えるのか？</b>（クリックで開く）</summary>

第6章で見たとおり、文字番号 `x` を one-hot ベクトル（`x` 番目だけ `1`、残り2060個は `0`）にして行列 `W` を右から掛けると、ちょうど `W` の `x` 行目がそのまま取り出されます。

```python no-run
# 数学的にはこの2つは完全に同じ計算！
# 方法A（第6章の行列積）：巨大な one-hot 行列 [B, 2061] を作って W と掛ける（0の掛け算が多くて遅い）
#   logits = F.one_hot(x, num_classes=V).float() @ W
# 方法B（この章の書き方）：直接 x 行目を取り出すだけ（同じ結果で圧倒的に速く、自動微分もそのまま動く）
#   logits = W[x]
```

そして初期値を全部 `0` にすると、すべての文字で $e^0 = 1$ になるため、softmax 後の確率はちょうど $\dfrac{1}{1 + 1 + \dots + 1} = \dfrac{1}{V}$ の完全な一様分布になります。

</details>

#### 📝 確認問題（`ch08-q02` · Lv.2 · W[x] の行取り出しと one-hot × W の数学的一致）

ニューラルbigramでは、形 `[V, V]` のロジットの表 `W` から `logits = W[x]` で次の文字の点数を取り出しています。これは第6章で学んだ行列の掛け算とどのような関係にありますか？

- **a.** `W[x]` のような整数インデックス参照は不連続な操作と見なされるため、PyTorch の自動微分（`backward()`）が動作せず勾配が一切流れない
- **b.** `W[x]` は `W` の全要素を列方向に平均してから `x` を掛ける計算なので、`one_hot(x) @ W` の行列積とはまったく異なる値になる
- **c.** `W[x]` は各行の合計が 1.0 になるよう内部で正規化した確率ベクトルを返すため、出力に対して `F.softmax` を掛ける必要がなくなる
- **d.** `x` を one-hot ベクトル（形 `[B, V]`）にして `one_hot(x) @ W` を計算するのと数学的に同じ結果になり、無駄な 0 の積を省いて高速に行を取り出せる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第6章で学んだ「one-hot ベクトル × 行列」が表のどの部分を取り出す操作だったか思い出しましょう。
- **ヒント 2**：`one_hot(x)` は `x` 番目だけ `1`・他はすべて `0` です。これを行列 `W` に左から掛けると、ちょうど `W` の `x` 行目がそのまま出てきます。
- **正解**：`d`

**考え方**：ニューラルbigramは「入力を one-hot にして1層の線形変換（重み行列 `W`）に通す最小のニューラルネットワーク」そのものです。

1. `one_hot(x)` は `x` 番目だけ `1`、残り $V-1$ 個が `0` のベクトル。
1. `one_hot(x) @ W` を計算すると、`x` 行目に `1`、他の行に `0` を掛けて足すので、`W[x]` そのままになる。
1. コードでは巨大な one-hot 行列を作らずに `W[x]` と書くことで高速かつ省メモリに実行できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q02

</details>

#### 📝 確認問題（`ch08-q03` · Lv.2 · ニューラルbigramのパラメータ数 V × V の計算）

語彙数が $V = 100$ 文字のテキストでニューラルbigramのロジット表 `W = torch.zeros((V, V))` を作りました。このモデルのパラメータ数はいくつですか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：2次元テンソル `W` の形 `[V, V]` に含まれる全要素数がモデルのパラメータ数になります。
- **ヒント 2**：行数が `100`、列数も `100` なので、表全体のマス目 `100 × 100` を計算しましょう。
- **正解**：`10000`

**考え方**：直前の文字（$V$ 通り）それぞれに対して次の文字（$V$ 通り）のロジットを持つため、パラメータ数は $V^2$ です。

1. `W.shape` は `[100, 100]`。
1. パラメータ数 `W.numel()` は $100 \times 100 = 10000$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q03

</details>

## 損失：交差エントロピー

第5章の損失（正解の確率の $-\log$ の平均）を、テンソルで計算してみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def loss_by_hand(W, x, y):
    logits = W[x]
    probs = F.softmax(logits, dim=1)
    p_correct = probs[torch.arange(len(y)), y]   # 各行の「正解の文字」の確率
    return -p_correct.log().mean()

print("手作りの損失：", loss_by_hand(W, xs[:1000], ys[:1000]).item())
print("F.cross_entropy：", F.cross_entropy(W[xs[:1000]], ys[:1000]).item())
print("log V：", math.log(V))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
手作りの損失： 7.630945205688477
F.cross_entropy： 7.630943775177002
log V： 7.630946580890459
```

</details>

> **💡 理論のポイント：交差エントロピー：`F.cross_entropy` が1つに融合している3ステップ**
>
> `F.cross_entropy(logits, targets)` は、次の **3つの計算** を内部でひとまとめ（かつオーバーフロー・アンダーフローしない数値的に安全な方法）で実行してくれます。
>
> 1. **安全な softmax**：生の点数 `logits`（形 `[B, V]`）の各行から最大値を引いて確率分布 $p$ に変換（第3章）
> 2. **正解の驚き $-\log(p[\text{target}])$**：各サンプル $i$ について、正解文字 `targets[i]` の確率だけを取り出して $-\log$ をとる（第5章）
> 3. **バッチ平均**：$B$ 個のサンプルの平均をとって1つのスカラー（損失）を返す
>
> $$\text{Loss} = -\frac{1}{B} \sum_{i=1}^{B} \log \left( \text{softmax}(\text{logits}_i)_{\text{target}_i} \right)$$
>
> まっさらなモデルの損失は $\log V = 7.631$。第5章の記録表の「でたらめ」と同じです。ここから、どこまで下げられるでしょう？

#### ⚠️ つまずき注意：絶対に注意！ `F.cross_entropy` には「softmax前の生のロジット」を渡す

初心者が最もやりがちなバグが、**すでに `F.softmax` を掛けた確率 `probs` を `F.cross_entropy` に渡してしまう**ことです。`F.cross_entropy` の内部でもう一度 softmax が掛かってしまい（**二重softmax**）、エラーも出ないまま学習が壊れます！

```python no-run
import math
# 正解がクラス0で、モデルがロジット [5.0, 0.0, 0.0]（ほぼ確信！）を出した例：
# ① 正しい使い方（生のロジットを渡す）：
#    softmax([5, 0, 0]) ≒ [0.987, 0.007, 0.007] → 損失 = -log(0.987) ≒ 0.013（ほぼ完璧！）
# ② 誤った使い方（確率 [0.987, 0.007, 0.007] をさらに F.cross_entropy に渡す）：
#    もう一度 softmax されて [0.570, 0.215, 0.215] → 損失 = -log(0.570) ≒ 0.562（大外れ扱い！）
```

**鉄則**：モデルの最後は必ず生のロジットのまま返し、`F.cross_entropy(logits, y)` に直接渡しましょう。確率が欲しいとき（生成や上位表示）だけ `F.softmax` を掛けます。

> **🐍 Python・PyTorchのポイント：2つのリストで同時に選ぶ**
>
> `probs[torch.arange(len(y)), y]` は、「0行目の `y[0]` 列、1行目の `y[1]` 列、2行目の `y[2]` 列……」を一度に取り出す書き方です（第6章の `P[xs, ys]` と同じ）。`torch.arange(n)` は `0, 1, 2, …, n-1` を作ります。

#### 📝 確認問題（`ch08-q04` · Lv.2 · F.cross_entropy が融合する3処理と「生のロジットを渡す」鉄則）

`F.cross_entropy(logits, targets)` が内部でひとまとめに実行している3つの処理と、第1引数に渡すべきものの組み合わせとして**正しい**のはどれですか？

- **a.** ①予測値と正解の二乗誤差 → ②平方根 → ③バッチ合計。第1引数には必ず事前に `F.softmax` を掛けた確率を渡す
- **b.** ①自動微分による勾配計算（`backward()`）→ ②重み更新 → ③勾配リセット。第1引数にはモデル本体を渡す
- **c.** ①安全な softmax → ②正解確率の $-\log$ → ③バッチ平均。第1引数には必ず **softmax 前の生のロジット** を渡す
- **d.** ①正解クラスの確率の $-\log$ と平均のみを実行する。そのため第1引数には必ず `F.softmax` 後の確率を渡す必要がある

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第5章で学んだ交差エントロピー損失を手計算した3手順と、`F.cross_entropy` が数値安定性のためにどこから計算を引き受けるか思い出しましょう。
- **ヒント 2**：`F.cross_entropy` は内部で `log_softmax` を実行してから正解列を取り出してバッチ平均します。だから引数に渡すのは確率ではなく生の点数です。
- **正解**：`c`

**考え方**：softmax と log を別々に計算すると非常に小さい確率で $\log(0) = -\infty$ の数値不安定が起きやすいため、`F.cross_entropy` は生のロジットから一気に安定計算する設計になっています。

1. ステップ1：生のロジット `logits`（形 `[B, V]`）に安全な `log_softmax` を適用。
1. ステップ2：各行 $i$ について正解番号 `targets[i]` の値を取り出し、符号を反転（$-\log p_i$）。
1. ステップ3：バッチ $B$ 個ぶんの平均をスカラーとして返す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q04

</details>

#### 📝 確認問題（`ch08-q05` · Lv.2 · 初期化直後の理想損失 log V の計算）

語彙が $V = 100$ 文字のモデルを、ロジットが全部 0 の状態から学習し始めました。最初の交差エントロピー損失はおよそいくつになるはずですか？（参考：$\log 10 \approx 2.303$、小数第2位までで入力、誤差 $\pm 0.05$ まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ロジットが全部 0 のとき全文字が等確率 $1/V$ になります。交差エントロピー損失 $-\log(1/V) = \log V$ の性質を使いましょう。
- **ヒント 2**：$100 = 10^2$ なので、対数の性質より $\log 100 = 2 \times \log 10$ です。参考値 $2.303$ を2倍してみてください。
- **正解**：`4.61`

**考え方**：すべての文字の確率が $1/V$ のとき、どの文字が正解であっても $-\log(1/V) = \log V$ になります。

1. ロジットが全部 0 → どの文字の確率も $p = 1 / 100 = 0.01$。
1. 交差エントロピー損失：$-\log(1 / 100) = \log 100$。
1. $\log 100 = 2 \times \log 10 \approx 2 \times 2.3026 \approx 4.605$（小数第2位で `4.61`）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q05

</details>

*(この節の追加問題 `ch08-q06`, `ch08-q07`, `ch08-q08` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html で自動採点・復習できます)*

## 学習：ミニバッチで坂を下る

いよいよ学習です。14万個の問題を毎回全部解くと時間がかかるので、**毎回ランダムに1024問だけ**選んで損失を計算し、坂を下ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
g = torch.Generator().manual_seed(0)
W = torch.zeros((V, V), requires_grad=True)
lr = 50
history = []

for step in range(3001):
    ix = torch.randint(0, len(xs), (1024,), generator=g)   # ランダムに1024問選ぶ（(1024,) は形の指定。要素1つでもカンマが必要）
    loss = F.cross_entropy(W[xs[ix]], ys[ix])               # ① 損失
    W.grad = None                                            # （勾配をリセット）
    loss.backward()                                          # ② 勾配
    with torch.no_grad():
        W -= lr * W.grad                                     # ③ 更新
    if step % 250 == 0:
        with torch.no_grad():
            val_loss = F.cross_entropy(W[vxs], vys).item()
        history.append((step, loss.item(), val_loss))
        if step % 500 == 0:
            print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss:.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ    0：訓練 7.631  検証 7.531
ステップ  500：訓練 4.196  検証 4.206
ステップ 1000：訓練 3.963  検証 3.963
ステップ 1500：訓練 3.686  検証 3.847
ステップ 2000：訓練 3.685  検証 3.771
ステップ 2500：訓練 3.548  検証 3.719
ステップ 3000：訓練 3.379  検証 3.685
```

</details>

CPUだと1〜2分かかります。損失がどんどん下がっていきます！ 7.63（まっさら）から、3.7あたりまで来ました。

ステップ0の検証損失だけ7.531と、$\log V = 7.631$ より少し小さいのは、検証損失を**1回更新した後**に測っているからです（コードでは、③の更新の後に検証しています）。訓練損失の7.631は更新の前に測っているので、ぴったり $\log V$ です。

> **🐍 Python・PyTorchのポイント：W.grad = None**
>
> 第7章では `.grad.zero_()` で勾配を0にしました。`W.grad = None` は「勾配を捨てる」で、効果は同じです（次の `backward()` で新しく作られます）。メモリの節約になるので、こちらがよく使われます。

<details><summary><b>基礎の補足：ニューラル学習ループ5ステップの解剖とミニバッチの読み方</b>（クリックで開く）</summary>

この先の章（MLP・GPT・ファインチューニング）でも、学習ループの心臓部は常に次の **5ステップ** です：

| 順番 | ステップ名 | この章のコード | 意味 |
|---|---|---|---|
| ① | **順伝播（forward）** | `logits = W[xs[ix]]` | ミニバッチ1024問の生の点数（形 `[1024, 2061]`）を出す |
| ② | **損失（loss）** | `loss = F.cross_entropy(logits, ys[ix])` | 正解とのずれを1つの数にまとめる |
| ③ | **勾配リセット（zero_grad）** | `W.grad = None` | 前回の勾配を消す（`backward()` の直前に置くのが安全！） |
| ④ | **逆伝播（backward）** | `loss.backward()` | 計算グラフを逆走し、全マスの勾配 `W.grad` を一括計算 |
| ⑤ | **更新（step）** | `with torch.no_grad(): W -= lr * W.grad` | 記録を切って勾配の逆向きへ一歩下る |

また `ix = torch.randint(0, len(xs), (1024,), generator=g)` のポイントも押さえておきましょう：
- `0` 以上 `len(xs)` 未満のランダムな行番号を `batch_size = 1024` 個引きます。
- Pythonでは要素1個のタプルには末尾カンマが必要なので、`(1024)` ではなく **`(1024,)`** と書きます（`(1024)` だとただの整数になりエラー）。
- 1エポック（訓練データ1周）＝ `len(xs) / batch_size` ステップなので、総エポック数は `(ステップ数 × batch_size) ÷ len(xs)` で計算できます。

</details>

> **💡 理論のポイント：ミニバッチ確率的勾配降下法（SGD）**
>
> 全部のデータで損失を計算する代わりに、ランダムに選んだ一部（**ミニバッチ**、今回は1024個）で計算して坂を下る方法を、**確率的勾配降下法**（stochastic gradient descent、**SGD**）と言います。
>
> - 一部だけなので、損失や勾配は少し**ぶれる**（上の「訓練」の値がガタガタなのはこのため）
> - でも計算が軽いので、同じ時間で**何倍も多く**坂を下れる
> - ぶれは、平均すると正しい方向を向いている
>
> 全問題を解いてから先生に採点してもらうより、10問ずつ解いてはその場で答え合わせをするほうが、早く上達しますよね。データ全体を1周することを **1エポック**（epoch）と呼びます。今回は1024問 × 3000回 ≒ 300万問なので、約21エポックです。
>
> 学習率が50と大きいのは、この表の1マスあたりの勾配がとても小さいからです。モデルごとに、ちょうどいい学習率は違います。

#### 📝 確認問題（`ch08-q09` · Lv.2 · ニューラル学習ループ5ステップ（forward → loss → zero_grad → backward → step））

ニューラルネットワークの1ステップの学習ループとして、**正しい順番**はどれですか？

- **a.** ①順伝播（ロジット計算）→ ②更新（`no_grad` 内で `W -= lr * W.grad`）→ ③損失計算（`F.cross_entropy`）→ ④勾配リセット → ⑤逆伝播（`loss.backward()`）
- **b.** ①順伝播（ロジット計算）→ ②損失計算（`F.cross_entropy`）→ ③勾配リセット（`W.grad = None`）→ ④逆伝播（`loss.backward()`）→ ⑤更新（`no_grad` 内で `W -= lr * W.grad`）
- **c.** ①勾配リセット（`W.grad = None`）→ ②更新（`W -= lr * W.grad`）→ ③順伝播（ロジット計算）→ ④損失計算（`F.cross_entropy`、`backward()` は不要）
- **d.** ①逆伝播（`loss.backward()`）→ ②勾配リセット（`W.grad = None`）→ ③更新（`no_grad` 内で `W -= lr * W.grad`）→ ④順伝播（ロジット計算）→ ⑤損失計算

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず予測して採点（損失計算）しないと勾配は測れません。また、前回の勾配を消すタイミングが `backward()` の前後どちらであるべきか考えましょう。
- **ヒント 2**：「順伝播 → 損失計算 → 前回の勾配リセット → `backward()` で今回の勾配計算 → `no_grad` 内で更新」の5段階をたどります。
- **正解**：`b`

**考え方**：勾配リセットは「`backward()` の前」か「`step()` の後」に置きます。`backward()` と `step()` の間に置いてしまうと勾配が消えてしまうので絶対NGです。

1. ① 順伝播：`logits = W[xs[ix]]`。
1. ② 損失：`loss = F.cross_entropy(logits, ys[ix])`。
1. ③ 勾配リセット：`W.grad = None`（`W.grad.zero_()` より省メモリ）。
1. ④ 逆伝播：`loss.backward()`。
1. ⑤ 更新：`with torch.no_grad(): W -= lr * W.grad`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q09

</details>

#### 📝 確認問題（`ch08-q10` · Lv.2 · ミニバッチSGDの利点と torch.randint の (1024,) の意味）

全14万件のデータで毎回損失を計算する代わりに、毎回ランダムに1024件だけ選ぶ **ミニバッチ確率的勾配降下法（SGD）** を使う最大の理由はどれですか？

- **a.** データを 1024 件に絞ることで各ステップの訓練損失のぶれが完全にゼロになり、検証損失とまったく同じ一直線の曲線で下がるから
- **b.** PyTorch の `F.cross_entropy` は仕様上 1 回の呼び出しで最大 1024 件までしか入力を受け付けず、それ以上渡すとエラーになるから
- **c.** 1ステップの計算が圧倒的に軽くなり、同じ時間で何十〜何百倍もの回数パラメータを更新できるから（個々のぶれは平均すると正しい方向を向く）
- **d.** 最初にランダム抽出した 1024 件だけを繰り返し暗記すれば、残りの約 14 万件は一度も学習に使う必要がなくなり時間が省けるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：全 14 万問をすべて解き終えてから 1 回だけ復習するのと、1024 問解くたびにすぐ復習するのとで、同じ時間での成長速度を比べてみましょう。
- **ヒント 2**：ミニバッチごとの勾配には多少のノイズがありますが、1 回の計算コストが約 1/140 になるため、圧倒的に多い更新回数で速く坂を下れます。
- **正解**：`c`

**考え方**：ミニバッチSGDは、GPUメモリに収まる範囲で並列計算の効率を最大化し、高速に何度も坂を下るための現代深層学習の標準手法です。

1. 全データ（14万件）で1回更新する時間で、ミニバッチ（1024件）なら100回以上更新できる。
1. ミニバッチごとの勾配には多少のノイズ（ぶれ）があるが、期待値は全データの勾配と一致する。
1. 結果として、はるかに短い時間で低い損失に到達できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q10

</details>

*(この節の追加問題 `ch08-q11` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html で自動採点・復習できます)*

## 学習曲線を描く

損失の変化をグラフにしたものを **学習曲線** と呼びます。学習がうまくいっているかを診断する、健康診断書のようなものです。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

steps = [h[0] for h in history]
plt.figure(figsize=(6, 3.5))
plt.plot(steps, [h[1] for h in history], label="訓練（ミニバッチ）")
plt.plot(steps, [h[2] for h in history], label="検証")
plt.axhline(3.581, color="gray", linestyle="--", label="数えたbigram（第5章）")
plt.xlabel("ステップ")
plt.ylabel("損失")
plt.ylim(3, 8)
plt.legend()
plt.title("ニューラルbigramの学習曲線")
plt.show()

![ニューラルbigramの学習曲線。損失が7.6から急降下し、3.7付近まで下がって、点線の3.58に近づいていく](https://sakunokyumei.github.io/llm-research-textbook/img/figs/08-neural-bigram/cell7_0.png)

検証損失は、灰色の点線（第5章で数えたbigramの3.581）に向かって、ゆっくり近づいています。

#### 📝 確認問題（`ch08-q12` · Lv.2 · 学習曲線の読み方（訓練のぶれ・検証の収束・lrが大きすぎる兆候））

ニューラルbigramの学習曲線で、訓練損失の線が検証損失の線よりも**ガタガタ上下にぶれている**主な理由はどれですか？

- **a.** 過学習を防ぐために、訓練データには毎ステップ自動でランダムなノイズ文字を混ぜているが、検証データにはノイズを混ぜていないから
- **b.** 学習率 `lr` の設定が極端に小さすぎるため、勾配が正しい下り方向を定められず毎ステップ振動してしまっているから
- **c.** 学習ループ内の勾配リセット（`W.grad = None`）が抜けているバグにより、前回の勾配が累積して学習が失敗しているから
- **d.** 訓練損失はそのステップでランダムに選んだ 1024 件のミニバッチだけで計算しているが、検証損失は検証データ全体（約 1.6 万件）で毎回計算しているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：毎回違う問題が出る小テストの点数と、毎回まったく同じ問題で受ける全範囲模試の点数を比べたとき、どちらが上下にぶれやすいでしょうか？
- **ヒント 2**：訓練損失は毎ステップ抽出し直す 1024 件のミニバッチで測っているのに対し、検証損失は毎回同じ全検証データ 16,246 件で測っています。
- **正解**：`d`

**考え方**：ミニバッチごとの訓練損失は「たまたま簡単な箇所を引いた回／難しい箇所を引いた回」で上下しますが、検証損失は全件固定で測るため滑らかな曲線になります。

1. 訓練損失：毎回ランダムな1024問で算出 → サンプルの引きで上下にぶれる。
1. 検証損失：毎回同じ `vxs, vys` 全16,246問で算出 → なめらかに推移する。
1. 学習の進み具合を正しく見るには、検証損失のカーブと訓練の平均的な傾向を見る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q12

</details>

## 学習した表の中身を見る

学習した `W` が、本当に「数えた答え」に似ているか確かめましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
with torch.no_grad():
    p = F.softmax(W[stoi["先"]], dim=0)
top = torch.topk(p, 5)
print("学習したモデル：「先」の次は…")
for prob_, i in zip(top.values, top.indices):
    print(f"  {itos[i.item()]!r} {prob_.item():.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
学習したモデル：「先」の次は…
  '生' 0.895
  'へ' 0.026
  '刻' 0.020
  'で' 0.012
  'に' 0.004
```

</details>

数えたときは「生」が0.88でした（第4章）。学習では0.895。誰にも「数えろ」とは教えていないのに、**損失を小さくしようと坂を下っただけで、数えた答えにたどり着いた**のです。

> **💡 理論のポイント：最尤推定：学習と数えるは同じ答えになる**
>
> これは偶然ではありません。「訓練データの交差エントロピー（驚きの平均）を最小にするような確率」は、数学的に **数えた割合（頻度）** とぴったり一致することが証明できます。この「データが最もありそうになる（尤もらしい）パラメータを選ぶ」という考え方を **最尤推定**（さいゆうすいてい、maximum likelihood estimation）と言います。
>
> なぜ一致するのかというと、交差エントロピー損失の傾き（勾配）を計算すると、ちょうど「モデルの予測確率 − 実際の出現割合」という形になるからです。予測確率が実際の頻度とぴったり一致した瞬間に、傾きがゼロになって坂の下りが止まります。だから勾配降下法を繰り返すだけで、自然と数えた割合と同じ確率に落ち着くのです。
>
> つまり、
>
> - 数えるbigram ＝ 最尤推定の答えを**直接計算**したもの
> - ニューラルbigram ＝ 同じ答えを**坂を下って探した**もの
>
> 直接計算できるのは、bigramのような単純なモデルだけです。ニューラルネットワークを使えば、直接計算できないような複雑なモデルでも、同じ「坂を下る」方法で答えを探せます。それが、この先のすべての章でやることです。

> **💡 理論のポイント：スムージングと正則化**
>
> 数えるbigramでは、見たことのないペアに確率0をつけないよう、スムージングで回数に少し下駄をはかせました。ニューラルbigramでは、`W` が全部0（＝全文字が等確率）から出発し、学習ステップ数が有限なので、極端な確率0にはなりません。
>
> さらに積極的に「極端な値になるのを防ぐ」方法として、損失に「`W` の値が大きいほど罰を与える」項を足すやり方があります。これを **正則化**（せいそくか、regularization）と言い、スムージングと同じ役割を果たします（第10章で `weight_decay` として登場します）。

#### 📝 確認問題（`ch08-q13` · Lv.2 · 最尤推定：交差エントロピー最小化が「数えた頻度」と一致する理由）

ニューラルbigramを十分長く学習させると、softmax後の確率の表はどのような値に近づきますか？

- **a.** 訓練データで数えた実際の出現割合（頻度）に近づく（交差エントロピー最小化＝最尤推定の答えと一致するため）
- **b.** 勾配降下法はランダムなミニバッチを使うため、数えた出現割合とはまったく無関係な別解に収束する
- **c.** 各行で最もよく出る 1 文字だけが確率 `1.0` になり、他のすべての文字の確率は完全に `0.0` になる
- **d.** 学習が進むほどロジットの差が平均化され、どの文字もすべて等しく $1/V$ になる一様分布に近づく

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第4章・第5章で作った「数える bigram」と、第8章のニューラル bigram の予測確率を本文で見比べた結果を思い出しましょう。
- **ヒント 2**：交差エントロピーの勾配は「モデルの予測確率 − データの出現割合」になります。勾配が 0 になって止まるのはどんな状態でしょうか？
- **正解**：`a`

**考え方**：交差エントロピーの勾配が 0（坂の下りが止まる谷底）になるのは、モデルの予測確率がデータの実際の出現割合とぴったり一致したときです。

1. 交差エントロピー損失のロジットに対する勾配 ＝ `予測確率 p − 実際の頻度 q`。
1. 予測が低すぎれば勾配が負 → ロジットを増やして確率を上げる。
1. 予測が高すぎれば勾配が正 → ロジットを減らして確率を下げる。
1. `p == q` で勾配が 0 になり、数えた頻度と同じ答えでピタリと止まる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q13

</details>

#### 📝 確認問題（`ch08-q14` · Lv.2 · 数えるモデルのスムージングとニューラルモデルの正則化の関係）

第5章の数えるbigramで使った「スムージング」と、ニューラルモデルの「正則化（重み `W` が大きくなりすぎると罰を与える）」に共通する役割はどれですか？

- **a.** 訓練データに含まれる珍しい漢字や記号をすべてひらがなに変換し、テキスト表記のゆれを事前に統一しておくこと
- **b.** 出現回数の少ない文字を語彙表から自動的に削除して、モデルが扱う語彙数 $V$ を半分以下に圧縮すること
- **c.** 訓練データに極端に偏って確率が 0 や 1 に張り付くのを防ぎ、未知のデータに対してもほどよく控えめな確率分布を保つこと
- **d.** 学習ループの反復回数を大幅に削減し、勾配計算やパラメータ更新にかかる計算時間をほぼゼロに短縮すること

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第5章のスムージングは、未出現のペアの確率が 0 になるのを防ぎました。ニューラルモデルで重み `W` が大きくなりすぎると確率はどうなるでしょう？
- **ヒント 2**：ロジット `W` を 0 付近へ引き戻す力（正則化）が働くと、softmax 後の確率分布は極端な 0/1 から穏やかな一様分布側へ寄ります。
- **正解**：`c`

**考え方**：重みを 0 に近づける正則化（第10章の `weight_decay`）は、ロジットを穏やかにして確率が極端な 0 になるのを防ぐため、スムージングと同じ効果を持ちます。

1. スムージング：全ペアの回数に $+k$ を足して、未出現でも確率 0 を防ぐ。
1. 正則化（重み減衰）：`W` が大きくなりすぎるのに罰を与え、ロジットを 0 寄りに抑える。
1. ロジットが 0 に近いほど確率は一様分布に近づくため、両者は同じ「極端さの抑制」を果たす。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q14

</details>

## 生成してみる

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
g = torch.Generator().manual_seed(1)
ix = stoi["私"]
out = ["私"]
with torch.no_grad():
    for _ in range(60):
        p = F.softmax(W[ix], dim=0)
        ix = torch.multinomial(p, 1, generator=g).item()
        out.append(itos[ix])
print("".join(out))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私はかえってに、それは、ないの奥さんにした。先生の頭に私の…
```

</details>

数えたbigramと同じくらいの「なんとなく漱石」です。それもそのはず、中身がほぼ同じ表なのですから。

## 記録表

| モデル | 章 | 検証データの損失 |
|---|---|---|
| でたらめ（一様分布） | 5 | 7.631 |
| bigram（数える、k=0.01） | 5 | 3.581 |
| **ニューラルbigram（SGD 3000ステップ）** | **8** | **3.685** |

ニューラル版はまだ数えた版に追いついていません。ステップ数を増やせば近づきますが、**同じ「直前1文字だけ」の情報しか使っていない**以上、数えた版を大きく超えることはできません。超えるには、モデルに**もっと前の文字**を見せる必要があります。次の章でやりましょう。

> **🔬 やってみよう：学習率を変えてみよう**
>
> `lr = 50` を `lr = 5`、`lr = 200`、`lr = 1000` に変えて、学習曲線を描き直してみましょう。どうなるか予想してから！ 教科書の検証で実際に試した結果（3000ステップ後の検証損失）はこうでした。
>
> - **5**：4.41。ゆっくりすぎて、3000ステップでは全然下りきらない
> - **200**：3.58。意外にも、50より速く、なめらかに下がる。このモデルは単純なので、「ちょうどいい」範囲が思ったより広いのです。そして、数えた版（3.581）とほぼ同じところで止まります。同じ「直前1文字」の情報しか使っていない以上、そこが限界だからです
> - **1000**：5.57。さすがに大きすぎて、損失が上がったり下がったり暴れ、ちっとも下がらない
>
> 「ちょうどいい学習率」はやってみないと分からず、モデルによって違う。だから研究者は、いくつか試して学習曲線を見比べるのです。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **損失が `nan` になった**：学習率が大きすぎて発散しています。学習率を小さくしましょう。
> - **`RuntimeError: element 0 of tensors does not require grad`**：`W` を作るときに `requires_grad=True` を付け忘れています。
> - **`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.`**：更新の行（`W -= lr * W.grad`）が `with torch.no_grad():` の中に入っていません（字下げを確認）。
> - **損失がほとんど下がらない**：学習率が小さすぎるか、勾配のリセット（`W.grad = None`）の位置がずれています。上のコードと1行ずつ見比べましょう。
> - **とても遅い**：検証損失の計算（`W[vxs]`）は重いので、毎ステップではなく250ステップごとにしています。Colabのランタイムを「T4 GPU」にして、`W` を作る行を `W = torch.zeros((V, V), requires_grad=True, device="cuda")` に変え、`xs, ys, vxs, vys` もそれぞれ `xs = xs.to("cuda")` のようにGPUへ送ると速くなります。`W` は、作った後から `.to("cuda")` すると「学習するパラメータ」ではなくなり、`W.grad` が `None` のままになるので、作るときに `device=` で指定するのがコツです。GPUにした場合は、「生成してみる」のセルの `p = F.softmax(W[ix], dim=0)` を `p = F.softmax(W[ix], dim=0).cpu()` に変えます（サイコロの `generator` がCPU用なので、確率もCPUに戻す必要があります）。くわしくは第13章で本格的にやります。

#### 📝 確認問題（`ch08-q15` · Lv.3 · 数えれば一瞬のbigramをわざわざ勾配降下法で学習させた意味）

bigram だけなら数えるほうが一瞬で検証損失 3.581 に着くのに、この章でわざわざ数分かけて勾配降下法（3.685）で学習させた「本当の意義」を、第4章の組み合わせ爆発とつなげて自分の言葉で説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「数える方法」が第4章でぶつかった壁（文脈を伸ばしたときの問題）と、「勾配降下法で学ぶ枠組み」の拡張性を対比してみましょう。
- **ヒント 2**：①長い文脈では数える方法は組み合わせ爆発で破綻する、②勾配降下法は MLP や GPT へそのまま拡張できる、③正解が既知の bigram で学習の正しさを答え合わせできた、の3点で整理できます。
- **解答の目安**：直前1文字のbigramなら数えられますが、直前8文字や100文字を見ようとすると組み合わせが爆発して数える方法は使えなくなります。一方「損失を決めて勾配降下法で坂を下る」学習の枠組みなら、どんなに複雑なニューラルネットワークにもそのまま拡張できます。この章は、正解（数えた頻度）が分かっている問題で「勾配降下法だけで本当に正しい確率分布にたどり着けること」を答え合わせ付きで確かめるステップです。

**考え方**：「数える」は1文字文脈で行き止まりですが、「勾配降下法＋交差エントロピー」は第9章のMLP、第12〜13章のGPTへとそのまま続く王道ルートです。

1. 数える方法の限界：直前8文字なら $2061^8$ 通りで組み合わせ爆発（第4章）。
1. 学習の一般性：ロジットを出す仕組みをどんなネットワークに差し替えても同じ学習ループが動く。
1. この章の役割：答えが既知のbigramで、学習が最尤解（数えた頻度）へ収束することを実証した。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html#ex-ch08-q15

</details>

**✅ できたらチェック**

- [ ] 入力と正解が1文字ずれた「教師データ」を作った
- [ ] `F.cross_entropy` と手作りの損失が一致することを確かめた
- [ ] ミニバッチSGDで損失が7.63から3.7付近まで下がるのを見た
- [ ] 学習した表が、数えた答え（「先」→「生」0.88）に近づいたことを確かめた
- [ ] 記録表にニューラルbigramを追加した

> **📝 まとめ**
>
> - ニューラルbigram ＝ **ロジットの表 `W`** を、交差エントロピーが小さくなるように勾配降下法で調整したもの
> - **ミニバッチSGD**：ランダムな一部で損失と勾配を計算し、少しずつ下る
> - 損失最小化（**最尤推定**）の答えは、数えた頻度と一致する。学習は「数える」の一般化
> - 学習曲線で、訓練と検証の損失を見比べるのが診断の基本

**ボンゴ先生**：いよいよ次は、本物のニューラルネットワークだ。直前の**何文字も**を見て、次の文字を予想する。そのために、文字を「数の組」で表す **埋め込み** という魔法を使う。「先生」と「教授」が似ていることを、モデルが自分で見つけ出すようになる。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html
- **次の章**：[第9章 文脈を広げる：埋め込みと多層パーセプトロン](https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html)